In [3]:
from pathlib import Path
from datasets import load_dataset

# --- Config ---
N = 200  # total samples to label (increase to continue further)
LEGIBLE_DIR = Path("../data/partitions/clear")
ILLEGIBLE_DIR = Path("../data/partitions/unclear")
PROGRESS_FILE = Path("../data/partitions/progress.txt")

LEGIBLE_DIR.mkdir(parents=True, exist_ok=True)
ILLEGIBLE_DIR.mkdir(parents=True, exist_ok=True)

ds = load_dataset("deepcopy/MathWriting-human")
ds_val = ds["val"].select(range(N))

start_idx = 0
if PROGRESS_FILE.exists():
    start_idx = int(PROGRESS_FILE.read_text().strip())
    print(f"Resuming from sample {start_idx}")

print(f"Loaded {N} samples. Starting at index {start_idx}.")

Resuming from sample 200
Loaded 200 samples. Starting at index 200.


In [ ]:
from IPython.display import display, clear_output

for idx in range(start_idx, N):
    clear_output(wait=True)
    sample = ds_val[idx]
    print(f"=== Sample {idx + 1} / {N} ===")
    print(f"GT: {sample['latex'][:120]}")
    display(sample["image"])
    
    while True:
        choice = input("1 = Clear, 2 = Unclear, u = Undo, q = Quit: ").strip().lower()
        if choice in ("1", "2", "u", "q"):
            break
        print("Invalid input. Try again.")
    
    if choice == "q":
        PROGRESS_FILE.write_text(str(idx))
        print(f"Saved progress at sample {idx}. Run cell again to resume.")
        break
    elif choice == "u":
        if idx > start_idx:
            # Re-do previous sample by adjusting — but since we're in a for loop,
            # we can't go back. Save progress and tell user to re-run.
            PROGRESS_FILE.write_text(str(max(0, idx - 1)))
            print(f"Undo: re-run this cell to redo sample {idx - 1}.")
            break
    else:
        dest = LEGIBLE_DIR if choice == "1" else ILLEGIBLE_DIR
        other = ILLEGIBLE_DIR if choice == "1" else LEGIBLE_DIR
        sample["image"].save(dest / f"{idx:05d}.png")
        other_path = other / f"{idx:05d}.png"
        if other_path.exists():
            other_path.unlink()
        PROGRESS_FILE.write_text(str(idx + 1))
else:
    clear_output(wait=True)
    print(f"Done! All {N} samples labeled.")
    legible_count = len(list(LEGIBLE_DIR.glob("*.png")))
    illegible_count = len(list(ILLEGIBLE_DIR.glob("*.png")))
    print(f"Legible: {legible_count}, Illegible: {illegible_count}")

Done! All 200 samples labeled.
Legible: 188, Illegible: 12


In [ ]:
from pathlib import Path

mine = set(p.name for p in Path("../data/partitions/unclear").glob("*.png"))
martin = set(p.name for p in Path("../data/martin_partitions/unclear").glob("*.png"))

both_unclear = sorted(mine & martin)

print(f"My unclear: {len(mine)}")
print(f"Martin's unclear: {len(martin)}")
print(f"Both unclear: {len(both_unclear)}")
print(f"\nSamples marked unclear by both annotators:")
for name in both_unclear:
    print(f"  {name}")

In [ ]:
from pathlib import Path

my_clear = set(p.name for p in Path("../data/partitions/clear").glob("*.png"))
my_unclear = set(p.name for p in Path("../data/partitions/unclear").glob("*.png"))
martin_clear = set(p.name for p in Path("../data/martin_partitions/clear").glob("*.png"))
martin_unclear = set(p.name for p in Path("../data/martin_partitions/unclear").glob("*.png"))

all_samples = my_clear | my_unclear  # all labeled samples

easy = sorted((my_clear & martin_clear))        # clear by both
medium = sorted((my_clear & martin_unclear) | (my_unclear & martin_clear))  # clear by one, unclear by other
hard = sorted((my_unclear & martin_unclear))     # unclear by both

print(f"Easy   (clear by both):   {len(easy)}")
print(f"Medium (clear by one):    {len(medium)}")
print(f"Hard   (unclear by both): {len(hard)}")
print(f"Total: {len(easy) + len(medium) + len(hard)}")

print(f"\nMedium samples:")
for name in medium:
    print(f"  {name}")

print(f"\nHard samples:")
for name in hard:
    print(f"  {name}")

## Evaluation on Easy / Medium / Hard Partitions

In [ ]:
import torch
import numpy as np
import pickle
import sys
import csv
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

sys.path.insert(0, "models")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ARTIFACTS = Path("artifacts")

def normalized_edit_distance(s1, s2):
    if len(s1) == 0 and len(s2) == 0: return 0.0
    if len(s1) == 0 or  len(s2) == 0: return 1.0
    d = [[0] * (len(s2) + 1) for _ in range(len(s1) + 1)]
    for i in range(len(s1) + 1): d[i][0] = i
    for j in range(len(s2) + 1): d[0][j] = j
    for i in range(1, len(s1) + 1):
        for j in range(1, len(s2) + 1):
            cost = 0 if s1[i-1] == s2[j-1] else 1
            d[i][j] = min(d[i-1][j]+1, d[i][j-1]+1, d[i-1][j-1]+cost)
    return d[len(s1)][len(s2)] / max(len(s1), len(s2))

def resize_pad_grayscale(img_pil, target=256, pad_value=255):
    img = img_pil.convert("L")
    w, h = img.size
    scale = target / max(w, h)
    new_w, new_h = max(1, int(round(w * scale))), max(1, int(round(h * scale)))
    img_rs = img.resize((new_w, new_h), resample=Image.BICUBIC)
    canvas = Image.new("L", (target, target), color=pad_value)
    canvas.paste(img_rs, ((target - new_w) // 2, (target - new_h) // 2))
    return canvas

# 92-vocab tokenizer
with open(ARTIFACTS / "lstm_tokenizer92.pkl", "rb") as f:
    tokenizer = pickle.load(f)
VS = max(tokenizer.word_index.values()) + 1
START = VS - 2
END = VS - 1
inv_vocab = {v: k for k, v in tokenizer.word_index.items()}

def decode(seq):
    return "".join(inv_vocab.get(t, "") for t in seq if t not in (0, START, END))

# Load RSLoRA r=32 model
from vit_lora_lstm_attn import ViTLatexModelLoRA as LSTMModel
model = LSTMModel(vocab_size=VS, lora_r=32, use_rslora=True).to(DEVICE)
ckpt = torch.load(ARTIFACTS / "lstm_rslora_r32.pt", map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model"])
model.eval()
print(f"LSTM RSLoRA r=32 loaded. vocab={VS}, device={DEVICE}")

# Preprocess all 200 images
all_images = []
for s in ds_val:
    img = resize_pad_grayscale(s["image"], target=256)
    img = np.array(img, dtype=np.float32) / 255.0
    all_images.append(img)
all_images = torch.tensor(np.array(all_images), dtype=torch.float32).unsqueeze(1)

# Tokenize GT via 92-vocab tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
gt_raw = [s["latex"] for s in ds_val]
seqs = tokenizer.texts_to_sequences(gt_raw)
seqs = [[START] + s + [END] for s in seqs]
tokens_val = pad_sequences(seqs, padding="post")
gt_strings = [decode(t.tolist()) for t in tokens_val]

# Build partition index sets
def filename_to_idx(name):
    return int(name.replace(".png", ""))

partitions = {
    "Easy": [filename_to_idx(n) for n in easy],
    "Medium": [filename_to_idx(n) for n in medium],
    "Hard": [filename_to_idx(n) for n in hard],
}

print(f"Partitions: {', '.join(f'{k}={len(v)}' for k, v in partitions.items())}")

In [ ]:
results = {}

for partition_name, indices in partitions.items():
    if len(indices) == 0:
        print(f"{partition_name}: no samples, skipping.")
        results[partition_name] = (0, 0.0, 0)
        continue

    exact = 0
    total_ed = 0.0

    for i in indices:
        img = all_images[i:i+1].repeat(1, 3, 1, 1).to(DEVICE)
        gt = gt_strings[i]

        with torch.no_grad():
            pred_tokens = model.generate_beam(img, max_len=150, sos_idx=START, eos_idx=END, beam_size=5)

        pred = decode(pred_tokens)
        ed = normalized_edit_distance(pred, gt)
        if pred == gt:
            exact += 1
        total_ed += ed

    n = len(indices)
    results[partition_name] = (exact, total_ed / n, n)
    print(f"{partition_name} ({n}): Exact={exact/n:.2%} ({exact}/{n}), Avg ED={total_ed/n:.4f}")

# Save CSV
csv_path = Path("../data/partition_eval_results.csv")
csv_path.parent.mkdir(parents=True, exist_ok=True)
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["partition", "n", "exact_match_pct", "avg_edit_distance"])
    writer.writeheader()
    for part, (exact, avg_ed, n) in results.items():
        writer.writerow({
            "partition": part,
            "n": n,
            "exact_match_pct": round(exact / n * 100, 2) if n > 0 else "N/A",
            "avg_edit_distance": round(avg_ed, 4) if n > 0 else "N/A",
        })
print(f"\nSaved to {csv_path}")

In [ ]:
parts = [k for k, (_, _, n) in results.items() if n > 0]
exact_pcts = [results[k][0] / results[k][2] * 100 for k in parts]
avg_eds = [results[k][1] for k in parts]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Exact match bar chart
axes[0].bar(parts, exact_pcts, color=["#4CAF50", "#FF9800", "#F44336"])
axes[0].set_ylabel("Exact Match (%)")
axes[0].set_title("Exact Match by Difficulty")
for i, v in enumerate(exact_pcts):
    axes[0].text(i, v + 0.5, f"{v:.1f}%", ha="center", fontsize=10)

# Edit distance bar chart
axes[1].bar(parts, avg_eds, color=["#4CAF50", "#FF9800", "#F44336"])
axes[1].set_ylabel("Avg Edit Distance")
axes[1].set_title("Avg Edit Distance by Difficulty")
for i, v in enumerate(avg_eds):
    axes[1].text(i, v + 0.01, f"{v:.3f}", ha="center", fontsize=10)

fig.suptitle("LSTM RSLoRA r=32 (92-vocab) — Partition Eval", fontsize=13)
plt.tight_layout()
plt.savefig("../data/partition_bar_chart.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved chart to data/partition_bar_chart.png")